In [2]:
!pip install requests beautifulsoup4
import re
import json
from datetime import datetime
import requests
from bs4 import BeautifulSoup

def extraer_producto_tauret_bs4(url):
    """
    Realiza Web Scraping sobre un artículo de Tauret Computadores
    utilizando el método BeautifulSoup4 y Requests.
    """
    # 1. Configuración de cabeceras HTTP para simular un navegador real
    headers = {
        "User-Agent": (
            "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
            "AppleWebKit/537.36 (KHTML, like Gecko) "
            "Chrome/128.0.0.0 Safari/537.36"
        ),
        "Accept-Language": "es-CO,es;q=0.9,en;q=0.8",
        "Accept": "text/html,application/xhtml+xml,application/xml;q=0.9,*/*;q=0.8"
    }

    try:
        # 2. Petición HTTP GET al servidor de Tauret Computadores
        respuesta = requests.get(url, headers=headers, timeout=15)
        respuesta.raise_for_status()
    except requests.exceptions.RequestException as error:
        print(f"[ERROR] No fue posible conectar con la página: {error}")
        return None

    # 3. Análisis sintáctico (Parsing) del árbol HTML con BeautifulSoup
    soup = BeautifulSoup(respuesta.text, "html.parser")

    # Extracción del título principal desde la etiqueta <h1>
    etiqueta_h1 = soup.find("h1")
    titulo = etiqueta_h1.get_text(strip=True) if etiqueta_h1 else "No encontrado"

    # Obtención del texto estructurado por saltos de línea del DOM
    texto_dom = soup.get_text(separator="\n", strip=True)

    # Función auxiliar para buscar patrones específicos dentro del texto extraído
    def buscar_patron(expresion, texto, grupo=1, valor_defecto=None):
        coincidencia = re.search(expresion, texto, re.IGNORECASE)
        return coincidencia.group(grupo).strip() if coincidencia else valor_defecto

    # 4. Extracción y limpieza de variables comerciales y de inventario
    precio_str = buscar_patron(r"COP\s*([\d\.]+)", texto_dom)
    precio_cop = int(precio_str.replace(".", "")) if precio_str else None

    unidades_str = buscar_patron(r"(\d+)\s+unidades disponibles", texto_dom)
    unidades_disponibles = int(unidades_str) if unidades_str else 0

    disponibilidad = buscar_patron(r"Disponibilidad:\s*([^\n]+)", texto_dom, valor_defecto="En Stock")
    garantia = buscar_patron(r"Garantía:\s*([^\n]+)", texto_dom, valor_defecto="12 meses")

    # 5. Extracción de identificadores únicos del producto (EAN, SKU, P/N)
    codigo_ean = buscar_patron(r"EAN:\s*([0-9]+)", texto_dom)
    codigo_sku = buscar_patron(r"SKU:\s*([0-9A-Za-z-]+)", texto_dom)
    numero_parte = buscar_patron(r"P/N:\s*([0-9A-Za-z-]+)", texto_dom)

    # 6. Extracción de dimensiones físicas de empaque
    peso = buscar_patron(r"Peso:\s*([^\n]+)", texto_dom)
    ancho = buscar_patron(r"Ancho:\s*([^\n]+)", texto_dom)
    alto = buscar_patron(r"Alto:\s*([^\n]+)", texto_dom)
    largo = buscar_patron(r"Largo:\s*([^\n]+)", texto_dom)

    # 7. Extracción de especificaciones técnicas de la tarjeta gráfica
    modelo_gpu = buscar_patron(r"Modelo:\s*([^\n]+)", texto_dom)
    motor_grafico = buscar_patron(r"Motor gráfico:\s*([^\n]+)", texto_dom)
    rendimiento_ia = buscar_patron(r"Rendimiento en IA:\s*([^\n]+)", texto_dom)
    nucleos_cuda = buscar_patron(r"Núcleos CUDA:\s*([^\n]+)", texto_dom)
    frecuencia_oc = buscar_patron(r"Modo OC:\s*([^\n]+)", texto_dom)
    capacidad_vram = buscar_patron(r"Capacidad:\s*([^\n]+)", texto_dom)
    tipo_vram = buscar_patron(r"Tipo:\s*(GDDR[0-9A-Za-z]+)", texto_dom)
    bus_pci = buscar_patron(r"Bus:\s*([^\n]+)", texto_dom)
    fuente_recomendada = buscar_patron(r"Fuente de poder recomendada:\s*([^\n]+)", texto_dom)

    # 8. Estructuración del diccionario final de resultados
    datos_articulo = {
        "metodologia_scraping": "BeautifulSoup4 (html.parser) + Requests",
        "fecha_extraccion": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
        "comercio": "Tauret Computadores S.A.S.",
        "url_origen": url,
        "informacion_comercial": {
            "titulo": titulo,
            "precio_cop": precio_cop,
            "precio_formateado": f"COP ${precio_cop:,.0f}".replace(",", ".") if precio_cop else None,
            "iva_incluido": "IVA INCLUIDO" in texto_dom.upper(),
            "envio_gratis": "Envío Gratis" in texto_dom,
            "estado_stock": disponibilidad,
            "unidades_disponibles": unidades_disponibles,
            "garantia": garantia
        },
        "identificadores_catalogo": {
            "EAN": codigo_ean,
            "SKU": codigo_sku,
            "Part_Number_PN": numero_parte
        },
        "dimensiones_empaque": {
            "peso": peso,
            "ancho": ancho,
            "alto": alto,
            "largo": largo
        },
        "especificaciones_tecnicas": {
            "modelo": modelo_gpu,
            "motor_grafico": motor_grafico,
            "rendimiento_ia": rendimiento_ia,
            "nucleos_cuda": nucleos_cuda,
            "frecuencia_modo_oc": frecuencia_oc,
            "memoria_capacidad": capacidad_vram,
            "memoria_tipo": tipo_vram,
            "interfaz_bus": bus_pci,
            "fuente_poder_recomendada": fuente_recomendada
        }
    }

    return datos_articulo


if __name__ == "__main__":
    URL_OBJETIVO = (
        "https://tauretcomputadores.com/product/"
        "tarjeta-de-video-asus-tuf-gaming-geforce-rtx-5090-gddr7-32gb"
    )

    print("Iniciando extracción con BeautifulSoup en Tauret Computadores...")
    resultado = extraer_producto_tauret_bs4(URL_OBJETIVO)

    if resultado:
        # Mostrar los resultados estructurados en consola
        print(json.dumps(resultado, indent=4, ensure_ascii=False))

        # Guardar los resultados en un archivo JSON local
        nombre_archivo = "resultado_rtx5090_tauret.json"
        with open(nombre_archivo, "w", encoding="utf-8") as archivo_json:
            json.dump(resultado, archivo_json, indent=4, ensure_ascii=False)
        print(f"\n[OK] Datos exportados exitosamente en: {nombre_archivo}")


Iniciando extracción con BeautifulSoup en Tauret Computadores...
{
    "metodologia_scraping": "BeautifulSoup4 (html.parser) + Requests",
    "fecha_extraccion": "2026-11-08 17:27:05",
    "comercio": "Tauret Computadores S.A.S.",
    "url_origen": "https://tauretcomputadores.com/product/tarjeta-de-video-asus-tuf-gaming-geforce-rtx-5090-gddr7-32gb",
    "informacion_comercial": {
        "titulo": "Tarjeta Gráfica ASUS TUF Gaming · GeForce RTX 5090 · 32GB GDDR7 · Triple Ventilador",
        "precio_cop": 25000000,
        "precio_formateado": "COP $25.000.000",
        "iva_incluido": true,
        "envio_gratis": true,
        "estado_stock": "En stock · Despacho el próximo día hábil",
        "unidades_disponibles": 0,
        "garantia": "12 meses"
    },
    "identificadores_catalogo": {
        "EAN": "4711387890356",
        "SKU": "11223",
        "Part_Number_PN": "90YV0LY1-M0AA00"
    },
    "dimensiones_empaque": {
        "peso": "4.00 kg",
        "ancho": "43 cm",
        